<h1>Categorical data</h1>

<p>This is an introduction to pandas categorical data type, including a short comparison with R’s <code>factor</code>.</p>

<p><code>Categoricals</code> are a pandas data type corresponding to categorical variables in statistics. A categorical variable takes on a limited, and usually fixed, number of possible values (<code>categories</code>; <code>levels</code> in R). Examples are gender, social class, blood type, country affiliation, observation time or rating via Likert scales.</p>

<p>In contrast to statistical categorical variables, categorical data might have an order (e.g. ‘strongly agree’ vs ‘agree’ or ‘first observation’ vs. ‘second observation’), but numerical operations (additions, divisions, …) are not possible.</p>

<p>All values of categorical data are either in <code>categories</code> or <code>np.nan</code>. Order is defined by the order of <code>categories</code>, not lexical order of the values. Internally, the data structure consists of a <code>categories</code> array and an integer array of <code>codes</code> which point to the real value in
the <code>categories</code> array.</p>

<p>The categorical data type is useful in the following cases:</p>

<ul>

<li><p>A string variable consisting of only a few different values. Converting such a string variable to a categorical variable will save some memory, see <a href="https://pandas.pydata.org/docs/user_guide/categorical.html#categorical-memory">here</a>.</p></li>

<li><p>The lexical order of a variable is not the same as the logical order (“one”, “two”, “three”).
By converting to a categorical and specifying an order on the categories, sorting and min/max will use the logical order instead of the lexical order, see <a href="https://pandas.pydata.org/docs/user_guide/categorical.html#categorical-sort">here</a>.</p></li>

<li><p>As a signal to other Python libraries that this column should be treated as a categorical variable (e.g. to use suitable statistical methods or plot types).</p></li>

</ul>

<p>See also the <a href="https://pandas.pydata.org/docs/reference/arrays.html#api-arrays-categorical">API docs on categoricals</a>.</p>

# <h2>Object creation</h2>

## <h3>Series creation</h3>

<p>Categorical <code>Series</code> or columns in a <code>DataFrame</code> can be created in several ways:</p>

<p>By specifying <code>dtype="category"</code> when constructing a <code>Series</code>:</p>

In [2]:
import pandas as pd
import numpy as np

In [3]:
s = pd.Series(["a", "b", "c", "a"], dtype="category")

s

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

<p>By converting an existing <code>Series</code> or column to a <code>category</code> dtype:</p>

In [4]:
df = pd.DataFrame({"A": ["a", "b", "c", "a"]})

df["B"] = df["A"].astype("category")

df

,A,B
0,a,a
1,b,b
2,c,c
3,a,a


<p>By using special functions, such as <a href="https://pandas.pydata.org/docs/reference/api/pandas.cut.html#pandas.cut" title="pandas.cut"><code>cut()</code></a>, which groups data into discrete bins. See the <a href="https://pandas.pydata.org/docs/user_guide/reshaping.html#reshaping-tile-cut">example on tiling</a> in the docs.</p>

In [5]:
df = pd.DataFrame({"value": np.random.randint(0, 100, 20)})

labels = ["{0} - {1}".format(i, i + 9) for i in range(0, 100, 10)]

df["group"] = pd.cut(x=df.value, bins=range(0, 105, 10), right=False, labels=labels)

df.head(10)

,value,group
0,14,10 - 19
1,87,80 - 89
2,72,70 - 79
3,83,80 - 89
4,63,60 - 69
5,76,70 - 79
6,63,60 - 69
7,55,50 - 59
8,70,70 - 79
9,62,60 - 69


<p>By passing a <a href="https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html#pandas.Categorical" title="pandas.Categorical"><code>pandas.Categorical</code></a> object to a <code>Series</code> or assigning it to a <code>DataFrame</code>.</p>

In [6]:
raw_cat = pd.Categorical(
    ["a", "b", "c", "a"], categories=["b", "c", "d"], ordered=False
)

s = pd.Series(raw_cat)

s

0    NaN
1      b
2      c
3    NaN
dtype: category
Categories (3, object): ['b', 'c', 'd']

In [7]:
df = pd.DataFrame({"A": ["a", "b", "c", "a"]})

df["B"] = raw_cat

df

,A,B
0,a,NaN
1,b,b
2,c,c
3,a,NaN


<p>Categorical data has a specific <code>category</code> <a href="https://pandas.pydata.org/docs/user_guide/basics.html#basics-dtypes">dtype</a>:</p>

In [8]:
df.dtypes

A      object
B    category
dtype: object

## <h3>DataFrame creation</h3>

<p>Similar to the previous section where a single column was converted to categorical, all columns in a <code>DataFrame</code> can be batch converted to categorical either during or after construction.</p>

<p>This can be done during construction by specifying <code>dtype="category"</code> in the <code>DataFrame</code> constructor:</p>

In [9]:
df = pd.DataFrame({"A": list("abca"), "B": list("bccd")}, dtype="category")

df.dtypes

A    category
B    category
dtype: object

<p>Note that the categories present in each column differ; the conversion is done column by column, so only labels present in a given column are categories:</p>

In [10]:
df["A"]

0    a
1    b
2    c
3    a
Name: A, dtype: category
Categories (3, object): ['a', 'b', 'c']

In [11]:
df["B"]

0    b
1    c
2    c
3    d
Name: B, dtype: category
Categories (3, object): ['b', 'c', 'd']

<p>Analogously, all columns in an existing <code>DataFrame</code> can be batch converted using <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html#pandas.DataFrame.astype" title="pandas.DataFrame.astype"><code>DataFrame.astype()</code></a>:</p>

In [12]:
df = pd.DataFrame({"A": list("abca"), "B": list("bccd")})

df_cat = df.astype("category")

df_cat.dtypes

A    category
B    category
dtype: object

<p>This conversion is likewise done column by column:</p>

In [13]:
df_cat["A"]

0    a
1    b
2    c
3    a
Name: A, dtype: category
Categories (3, object): ['a', 'b', 'c']

In [14]:
df_cat["B"]

0    b
1    c
2    c
3    d
Name: B, dtype: category
Categories (3, object): ['b', 'c', 'd']

## <h3>Controlling behavior</h3>

<p>In the examples above where we passed <code>dtype='category'</code>, we used the default behavior:</p>

<ol>
<li><p>Categories are inferred from the data.</p></li>
<li><p>Categories are unordered.</p></li>
</ol>

<p>To control those behaviors, instead of passing <code>'category'</code>, use an instance of <code>CategoricalDtype</code>.</p>

In [15]:
from pandas.api.types import CategoricalDtype

s = pd.Series(["a", "b", "c", "a"])

cat_type = CategoricalDtype(categories=["b", "c", "d"], ordered=True)

s_cat = s.astype(cat_type)

s_cat

0    NaN
1      b
2      c
3    NaN
dtype: category
Categories (3, object): ['b' < 'c' < 'd']

<p>Similarly, a <code>CategoricalDtype</code> can be used with a <code>DataFrame</code> to ensure that categories are consistent among all columns.</p>

In [16]:
from pandas.api.types import CategoricalDtype

df = pd.DataFrame({"A": list("abca"), "B": list("bccd")})

cat_type = CategoricalDtype(categories=list("abcd"), ordered=True)

df_cat = df.astype(cat_type)

df_cat["A"]

0    a
1    b
2    c
3    a
Name: A, dtype: category
Categories (4, object): ['a' < 'b' < 'c' < 'd']

In [17]:
df_cat["B"]

0    b
1    c
2    c
3    d
Name: B, dtype: category
Categories (4, object): ['a' < 'b' < 'c' < 'd']

<div class="alert alert-block alert-info">
<p>Note</p>
<p>To perform table-wise conversion, where all labels in the entire <code>DataFrame</code> are used as categories for each column, the <code>categories</code> parameter can be determined programmatically by <code>categories = pd.unique(df.to_numpy().ravel())</code>.</p>
</div>

<p>If you already have <code>codes</code> and <code>categories</code>, you can use the
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Categorical.from_codes.html#pandas.Categorical.from_codes" title="pandas.Categorical.from_codes"><code>from_codes()</code></a> constructor to save the factorize step during normal constructor mode:</p>

In [18]:
splitter = np.random.choice([0, 1], 5, p=[0.5, 0.5])

s = pd.Series(pd.Categorical.from_codes(splitter, categories=["train", "test"]))

## <h3>Regaining original data</h3>

<p>To get back to the original <code>Series</code> or NumPy array, use <code>Series.astype(original_dtype)</code> or <code>np.asarray(categorical)</code>:</p>

In [19]:
s = pd.Series(["a", "b", "c", "a"])

s

0    a
1    b
2    c
3    a
dtype: object

In [20]:
s2 = s.astype("category")

s2

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [21]:
s2.astype(str)

0    a
1    b
2    c
3    a
dtype: object

In [22]:
np.asarray(s2)

array(['a', 'b', 'c', 'a'], dtype=object)

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>In contrast to R’s <code>factor</code> function, categorical data is not converting input values to strings; categories will end up the same data type as the original values.</p>

</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>In contrast to R’s <code>factor</code> function, there is currently no way to assign/change labels at creation time. Use <code>categories</code> to change the categories after creation time.</p>
</div>

# <h2>CategoricalDtype</h2>

<p>A categorical’s type is fully described by</p>

<ol>
<li><p><code>categories</code>: a sequence of unique values and no missing values</p></li>
<li><p><code>ordered</code>: a boolean</p></li>
</ol>

<p>This information can be stored in a <code>CategoricalDtype</code>. The <code>categories</code> argument is optional, which implies that the actual categories should be inferred from whatever is present in the data when the <a href="https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html#pandas.Categorical" title="pandas.Categorical"><code>pandas.Categorical</code></a> is created.
The categories are assumed to be unordered by default.</p>

In [23]:
from pandas.api.types import CategoricalDtype

CategoricalDtype(["a", "b", "c"])

CategoricalDtype(categories=['a', 'b', 'c'], ordered=False, categories_dtype=object)

In [24]:
CategoricalDtype(["a", "b", "c"], ordered=True)

CategoricalDtype(categories=['a', 'b', 'c'], ordered=True, categories_dtype=object)

In [25]:
CategoricalDtype()

CategoricalDtype(categories=None, ordered=False, categories_dtype=None)

<p>A <code>CategoricalDtype</code> can be used in any place pandas expects a <code>dtype</code>. For example <a href="https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html#pandas.read_csv" title="pandas.read_csv"><code>pandas.read_csv()</code></a>,
<a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html#pandas.DataFrame.astype" title="pandas.DataFrame.astype"><code>pandas.DataFrame.astype()</code></a>, or in the <code>Series</code> constructor.</p>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>As a convenience, you can use the string <code>'category'</code> in place of a
<code>CategoricalDtype</code> when you want the default behavior of the categories being unordered, and equal to the set values present in the array. In other words, <code>dtype='category'</code> is equivalent to <code>dtype=CategoricalDtype()</code>.</p>
</div>

## <h3>Equality semantics</h3>

<p>Two instances of <code>CategoricalDtype</code> compare equal
whenever they have the same categories and order. When comparing two
unordered categoricals, the order of the <code>categories</code> is not considered.</p>

In [26]:
c1 = CategoricalDtype(["a", "b", "c"], ordered=False)

# Equal, since order is not considered when ordered=False
c1 == CategoricalDtype(["b", "c", "a"], ordered=False)

True

In [27]:
# Unequal, since the second CategoricalDtype is ordered
In [51]: c1 == CategoricalDtype(["a", "b", "c"], ordered=True)

False

<p>All instances of <code>CategoricalDtype</code> compare equal to the string <code>'category'</code>.</p>

In [28]:
c1 == "category"

True

# <h2>Description</h2>

<p>Using <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html#pandas.DataFrame.describe" title="pandas.DataFrame.describe"><code>describe()</code></a> on categorical data will produce similar
output to a <code>Series</code> or <code>DataFrame</code> of type <code>string</code>.</p>

In [29]:
cat = pd.Categorical(["a", "c", "c", np.nan], categories=["b", "a", "c"])

df = pd.DataFrame({"cat": cat, "s": ["a", "c", "c", np.nan]})

df.describe()

,cat,s
count,3,3
unique,2,2
top,c,c
freq,2,2


In [30]:
df["cat"].describe()

count     3
unique    2
top       c
freq      2
Name: cat, dtype: object

# <h2>Working with categories</h2>

<p>Categorical data has a <code>categories</code> and a <code>ordered</code> property, which list their possible values and whether the ordering matters or not. These properties are exposed as <code>s.cat.categories</code> and <code>s.cat.ordered</code>. If you don’t manually
specify categories and ordering, they are inferred from the passed arguments.</p>

In [31]:
s = pd.Series(["a", "b", "c", "a"], dtype="category")

s.cat.categories

Index(['a', 'b', 'c'], dtype='object')

In [32]:
s.cat.ordered

False

<p>It’s also possible to pass in the categories in a specific order:</p>

In [33]:
s = pd.Series(pd.Categorical(["a", "b", "c", "a"], categories=["c", "b", "a"]))

s.cat.categories

Index(['c', 'b', 'a'], dtype='object')

In [34]:
s.cat.ordered

False

<div class="alert alert-block alert-info">
<p>Note</p>
<p>New categorical data are <strong>not</strong> automatically ordered. You must explicitly pass <code>ordered=True</code> to indicate an ordered <code>Categorical</code>.</p>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>The result of <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html#pandas.Series.unique" title="pandas.Series.unique"><code>unique()</code></a> is not always the same as <code>Series.cat.categories</code>, because <code>Series.unique()</code> has a couple of guarantees, namely that it returns categories in the order of appearance, and it only includes values that are actually present.</p>

In [35]:
s = pd.Series(list("babc")).astype(CategoricalDtype(list("abcd")))

In [36]:
s

0    b
1    a
2    b
3    c
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [37]:
# categories
s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='object')

In [38]:
# uniques
s.unique()

['b', 'a', 'c']
Categories (4, object): ['a', 'b', 'c', 'd']

# <h3>Renaming categories</h3>

<p>Renaming categories is done by using the <code>rename_categories()</code> method:</p>

In [39]:
s = pd.Series(["a", "b", "c", "a"], dtype="category")

In [40]:
s

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [41]:
new_categories = ["Group %s" % g for g in s.cat.categories]

s = s.cat.rename_categories(new_categories)

In [42]:
s

0    Group a
1    Group b
2    Group c
3    Group a
dtype: category
Categories (3, object): ['Group a', 'Group b', 'Group c']

In [43]:
# You can also pass a dict-like object to map the renaming
s = s.cat.rename_categories({1: "x", 2: "y", 3: "z"})

In [44]:
s

0    Group a
1    Group b
2    Group c
3    Group a
dtype: category
Categories (3, object): ['Group a', 'Group b', 'Group c']

<div class="alert alert-block alert-info">
<p>Note</p>
<p>In contrast to R’s <code>factor</code>, categorical data can have categories of other types than string.</p>
</div>

<p>Categories must be unique or a <code>ValueError</code> is raised:</p>

In [45]:
try:
    s = s.cat.rename_categories([1, 1, 1])
except ValueError as e:
    print("ValueError:", str(e))

ValueError: Categorical categories must be unique


<p>Categories must also not be <code>NaN</code> or a <code>ValueError</code> is raised:</p>

In [46]:
try:
    s = s.cat.rename_categories([1, 2, np.nan])
except ValueError as e:
    print("ValueError:", str(e))

ValueError: Categorical categories cannot be null


## <h3>Appending new categories</h3>

<p>Appending categories can be done by using the <code>add_categories()</code> method:</p>

In [47]:
s = s.cat.add_categories([4])

s.cat.categories

Index(['Group a', 'Group b', 'Group c', 4], dtype='object')

In [48]:
s

0    Group a
1    Group b
2    Group c
3    Group a
dtype: category
Categories (4, object): ['Group a', 'Group b', 'Group c', 4]

## <h3>Removing categories</h3>

<p>Removing categories can be done by using the <code>remove_categories()</code> method. Values which are removed are replaced by <code>np.nan</code>.:</p>

In [49]:
s = s.cat.remove_categories([4])

In [50]:
s

0    Group a
1    Group b
2    Group c
3    Group a
dtype: category
Categories (3, object): ['Group a', 'Group b', 'Group c']

## <h3>Removing unused categories</h3>

<p>Removing unused categories can also be done:</p>

In [53]:
s = pd.Series(pd.Categorical(["a", "b", "a"], categories=["a", "b", "c", "d"]))

In [54]:
s

0    a
1    b
2    a
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [55]:
s.cat.remove_unused_categories()

0    a
1    b
2    a
dtype: category
Categories (2, object): ['a', 'b']

## <h3>Setting categories</h3>

<p>If you want to do remove and add new categories in one step (which has some speed advantage), or simply set the categories to a predefined scale, use <code>set_categories()</code>.</p>

In [57]:
s = pd.Series(["one", "two", "four", "-"], dtype="category")

In [58]:
s

0     one
1     two
2    four
3       -
dtype: category
Categories (4, object): ['-', 'four', 'one', 'two']

In [59]:
s = s.cat.set_categories(["one", "two", "three", "four"])

In [60]:
s

0     one
1     two
2    four
3     NaN
dtype: category
Categories (4, object): ['one', 'two', 'three', 'four']

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Be aware that <code>Categorical.set_categories()</code> cannot know whether some category is omitted
intentionally or because it is misspelled or (under Python3) due to a type difference (e.g.,
NumPy S1 dtype and Python strings). This can result in surprising behaviour!</p>
</div>
</section>
</section>

# <h2>Sorting and order</h2>

<p id="categorical-sort">If categorical data is ordered (<code>s.cat.ordered == True</code>), then the order of the categories has a
meaning and certain operations are possible. If the categorical is unordered, <code>.min()/.max()</code> will raise a <code>TypeError</code>.</p>

In [64]:
s = pd.Series(pd.Categorical(["a", "b", "c", "a"], ordered=False))

In [65]:
s = s.sort_values()

In [66]:
s = pd.Series(["a", "b", "c", "a"]).astype(CategoricalDtype(ordered=True))

In [67]:
s = s.sort_values()

In [68]:
s

0    a
3    a
1    b
2    c
dtype: category
Categories (3, object): ['a' < 'b' < 'c']

In [69]:
s.min(), s.max()

('a', 'c')

<p>You can set categorical data to be ordered by using <code>as_ordered()</code> or unordered by using <code>as_unordered()</code>. These will by default return a <em>new</em> object.</p>

In [70]:
s.cat.as_ordered()

0    a
3    a
1    b
2    c
dtype: category
Categories (3, object): ['a' < 'b' < 'c']

In [71]:
s.cat.as_unordered()

0    a
3    a
1    b
2    c
dtype: category
Categories (3, object): ['a', 'b', 'c']

<p>Sorting will use the order defined by categories, not any lexical order present on the data type.
This is even true for strings and numeric data:</p>

In [72]:
s = pd.Series([1, 2, 3, 1], dtype="category")

In [73]:
s = s.cat.set_categories([2, 3, 1], ordered=True)

In [74]:
s

0    1
1    2
2    3
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [75]:
s = s.sort_values()

In [76]:
s

1    2
2    3
0    1
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [77]:
s.min(), s.max()

(2, 1)

## <h3>Reordering</h3>

<p>Reordering the categories is possible via the <code>Categorical.reorder_categories()</code> and the <code>Categorical.set_categories()</code> methods. For <code>Categorical.reorder_categories()</code>, all
old categories must be included in the new categories and no new categories are allowed. This will necessarily make the sort order the same as the categories order.</p>

In [78]:
s = pd.Series([1, 2, 3, 1], dtype="category")

In [79]:
s = s.cat.reorder_categories([2, 3, 1], ordered=True)

In [80]:
s

0    1
1    2
2    3
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [81]:
s = s.sort_values()

In [82]:
s

1    2
2    3
0    1
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [83]:
 s.min(), s.max()

(2, 1)

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Note the difference between assigning new categories and reordering the categories: the first renames categories and therefore the individual values in the <code>Series</code>, but if the first position was sorted last, the renamed value will still be sorted last. Reordering means that the way values are sorted is different afterwards, but not that individual values in the <code>Series</code> are changed.</p>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>If the <code>Categorical</code> is not ordered, <a href="../reference/api/pandas.Series.min.html#pandas.Series.min" title="pandas.Series.min"><code>Series.min()</code></a> and <a href="../reference/api/pandas.Series.max.html#pandas.Series.max" title="pandas.Series.max"><code>Series.max()</code></a> will raise <code>TypeError</code>. Numeric operations like <code>+</code>, <code>-</code>, <code>*</code>, <code>/</code> and operations based on them (e.g. <a href="../reference/api/pandas.Series.median.html#pandas.Series.median" title="pandas.Series.median"><code>Series.median()</code></a>, which would need to compute the mean between two values if the length of an array is even) do not work and raise a <code>TypeError</code>.</p>
</div>

## <h3>Multi column sorting</h3>

<p>A categorical dtyped column will participate in a multi-column sort in a similar manner to other columns.
The ordering of the categorical is determined by the <code>categories</code> of that column.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell31"><span><span class="gp">In [108]: <span class="n">dfs <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">(
<span class="gp">    <span class="p">{
<span class="gp">        <span class="s2">"A"<span class="p">: <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">(
<span class="gp">            <span class="nb">list<span class="p">(<span class="s2">"bbeebbaa"<span class="p">),
<span class="gp">            <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"e"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">],
<span class="gp">            <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">,
<span class="gp">        <span class="p">),
<span class="gp">        <span class="s2">"B"<span class="p">: <span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">2<span class="p">, <span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">1<span class="p">],
<span class="gp">    <span class="p">}
<span class="gp"><span class="p">)
<span class="gp">

<span class="gp">In [109]: <span class="n">dfs<span class="o">.<span class="n">sort_values<span class="p">(<span class="n">by<span class="o">=<span class="p">[<span class="s2">"A"<span class="p">, <span class="s2">"B"<span class="p">])
<span class="gh">Out[109]:
<span class="go">   A  B
<span class="go">2  e  1
<span class="go">3  e  2
<span class="go">7  a  1
<span class="go">6  a  2
<span class="go">0  b  1
<span class="go">5  b  1
<span class="go">1  b  2
<span class="go">4  b  2
</pre>
</div>
</div>

<p>Reordering the <code>categories</code> changes a future sort.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell32"><span><span class="gp">In [110]: <span class="n">dfs<span class="p">[<span class="s2">"A"<span class="p">] <span class="o">= <span class="n">dfs<span class="p">[<span class="s2">"A"<span class="p">]<span class="o">.<span class="n">cat<span class="o">.<span class="n">reorder_categories<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"e"<span class="p">])

<span class="gp">In [111]: <span class="n">dfs<span class="o">.<span class="n">sort_values<span class="p">(<span class="n">by<span class="o">=<span class="p">[<span class="s2">"A"<span class="p">, <span class="s2">"B"<span class="p">])
<span class="gh">Out[111]:
<span class="go">   A  B
<span class="go">7  a  1
<span class="go">6  a  2
<span class="go">0  b  1
<span class="go">5  b  1
<span class="go">1  b  2
<span class="go">4  b  2
<span class="go">2  e  1
<span class="go">3  e  2
</pre>
</div>
</div>

# <h2>Comparisons</h2>

<p>Comparing categorical data with other objects is possible in three cases:</p>

<ul>
<li><p>Comparing equality (<code>==</code> and <code>!=</code>) to a list-like object (list, Series, array, …) of the same length as the categorical data.</p></li>
<li><p>All comparisons (<code>==</code>, <code>!=</code>, <code>&gt;</code>, <code>&gt;=</code>, <code>&lt;</code>, and <code>&lt;=</code>) of categorical data to another categorical Series, when <code>ordered==True</code> and the <code>categories</code> are the same.</p></li>
<li><p>All comparisons of a categorical data to a scalar.</p></li>
</ul>

<p>All other comparisons, especially “non-equality” comparisons of two categoricals with different categories or a categorical with any list-like object, will raise a <code>TypeError</code>.</p>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Any “non-equality” comparisons of categorical data with a <code>Series</code>, <code>np.array</code>, <code>list</code> or
categorical data with different categories or ordering will raise a <code>TypeError</code> because custom categories ordering could be interpreted in two ways: one with taking into account the
ordering and one without.</p>
</div>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell33"><span><span class="gp">In [112]: <span class="n">cat <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">])<span class="o">.<span class="n">astype<span class="p">(<span class="n">CategoricalDtype<span class="p">([<span class="mi">3<span class="p">, <span class="mi">2<span class="p">, <span class="mi">1<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">))

<span class="gp">In [113]: <span class="n">cat_base <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="mi">2<span class="p">, <span class="mi">2<span class="p">, <span class="mi">2<span class="p">])<span class="o">.<span class="n">astype<span class="p">(<span class="n">CategoricalDtype<span class="p">([<span class="mi">3<span class="p">, <span class="mi">2<span class="p">, <span class="mi">1<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">))

<span class="gp">In [114]: <span class="n">cat_base2 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="mi">2<span class="p">, <span class="mi">2<span class="p">, <span class="mi">2<span class="p">])<span class="o">.<span class="n">astype<span class="p">(<span class="n">CategoricalDtype<span class="p">(<span class="n">ordered<span class="o">=<span class="kc">True<span class="p">))

<span class="gp">In [115]: <span class="n">cat
<span class="gh">Out[115]:
<span class="go">0    1
<span class="go">1    2
<span class="go">2    3
<span class="go">dtype: category
<span class="go">Categories (3, int64): [3 &lt; 2 &lt; 1]

<span class="gp">In [116]: <span class="n">cat_base
<span class="gh">Out[116]:
<span class="go">0    2
<span class="go">1    2
<span class="go">2    2
<span class="go">dtype: category
<span class="go">Categories (3, int64): [3 &lt; 2 &lt; 1]

<span class="gp">In [117]: <span class="n">cat_base2
<span class="gh">Out[117]:
<span class="go">0    2
<span class="go">1    2
<span class="go">2    2
<span class="go">dtype: category
<span class="go">Categories (1, int64): [2]
</pre>
</div>
</div>

<p>Comparing to a categorical with the same categories and ordering or to a scalar works:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell34"><span><span class="gp">In [118]: <span class="n">cat <span class="o">&gt; <span class="n">cat_base
<span class="gh">Out[118]:
<span class="go">0     True
<span class="go">1    False
<span class="go">2    False
<span class="go">dtype: bool

<span class="gp">In [119]: <span class="n">cat <span class="o">&gt; <span class="mi">2
<span class="gh">Out[119]:
<span class="go">0     True
<span class="go">1    False
<span class="go">2    False
<span class="go">dtype: bool
</pre>
</div>
</div>

<p>Equality comparisons work with any list-like object of same length and scalars:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell35"><span><span class="gp">In [120]: <span class="n">cat <span class="o">== <span class="n">cat_base
<span class="gh">Out[120]:
<span class="go">0    False
<span class="go">1     True
<span class="go">2    False
<span class="go">dtype: bool

<span class="gp">In [121]: <span class="n">cat <span class="o">== <span class="n">np<span class="o">.<span class="n">array<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">])
<span class="gh">Out[121]:
<span class="go">0    True
<span class="go">1    True
<span class="go">2    True
<span class="go">dtype: bool

<span class="gp">In [122]: <span class="n">cat <span class="o">== <span class="mi">2
<span class="gh">Out[122]:
<span class="go">0    False
<span class="go">1     True
<span class="go">2    False
<span class="go">dtype: bool
</pre>
</div>
</div>

<p>This doesn’t work because the categories are not the same:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell36"><span><span class="gp">In [123]: <span class="k">try<span class="p">:
<span class="gp">    <span class="n">cat <span class="o">&gt; <span class="n">cat_base2
<span class="gp"><span class="k">except <span class="ne">TypeError <span class="k">as <span class="n">e<span class="p">:
<span class="gp">    <span class="nb">print<span class="p">(<span class="s2">"TypeError:"<span class="p">, <span class="nb">str<span class="p">(<span class="n">e<span class="p">))
<span class="gp">
<span class="go">TypeError: Categoricals can only be compared if 'categories' are the same.
</pre>
</div>
</div>

<p>If you want to do a “non-equality” comparison of a categorical series with a list-like object which is not categorical data, you need to be explicit and convert the categorical data back to the original values:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell37"><span><span class="gp">In [124]: <span class="n">base <span class="o">= <span class="n">np<span class="o">.<span class="n">array<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">])

<span class="gp">In [125]: <span class="k">try<span class="p">:
<span class="gp">    <span class="n">cat <span class="o">&gt; <span class="n">base
<span class="gp"><span class="k">except <span class="ne">TypeError <span class="k">as <span class="n">e<span class="p">:
<span class="gp">    <span class="nb">print<span class="p">(<span class="s2">"TypeError:"<span class="p">, <span class="nb">str<span class="p">(<span class="n">e<span class="p">))
<span class="gp">
<span class="go">TypeError: Cannot compare a Categorical for op __gt__ with type &lt;class 'numpy.ndarray'&gt;.
<span class="go">If you want to compare values, use 'np.asarray(cat) &lt;op&gt; other'.

<span class="gp">In [126]: <span class="n">np<span class="o">.<span class="n">asarray<span class="p">(<span class="n">cat<span class="p">) <span class="o">&gt; <span class="n">base
<span class="gh">Out[126]: <span class="go">array([False, False, False])
</pre>
</div>
</div>

<p>When you compare two unordered categoricals with the same categories, the order is not considered:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell38"><span><span class="gp">In [127]: <span class="n">c1 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">False<span class="p">)

<span class="gp">In [128]: <span class="n">c2 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"b"<span class="p">, <span class="s2">"a"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">False<span class="p">)

<span class="gp">In [129]: <span class="n">c1 <span class="o">== <span class="n">c2
<span class="gh">Out[129]: <span class="go">array([ True,  True])
</pre>
</div>
</div>

# <h2>Operations</h2>

<p>Apart from <a href="../reference/api/pandas.Series.min.html#pandas.Series.min" title="pandas.Series.min"><code>Series.min()</code></a>, <a href="../reference/api/pandas.Series.max.html#pandas.Series.max" title="pandas.Series.max"><code>Series.max()</code></a> and <a href="../reference/api/pandas.Series.mode.html#pandas.Series.mode" title="pandas.Series.mode"><code>Series.mode()</code></a>, the
following operations are possible with categorical data:</p>

<p><code>Series</code> methods like <a href="../reference/api/pandas.Series.value_counts.html#pandas.Series.value_counts" title="pandas.Series.value_counts"><code>Series.value_counts()</code></a> will use all categories, even if some categories are not present in the data:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell39"><span><span class="gp">In [130]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">(<span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"c"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"c"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"d"<span class="p">]))

<span class="gp">In [131]: <span class="n">s<span class="o">.<span class="n">value_counts<span class="p">()
<span class="gh">Out[131]:
<span class="go">c    2
<span class="go">a    1
<span class="go">b    1
<span class="go">d    0
<span class="go">Name: count, dtype: int64
</pre>
</div>
</div>

<p><code>DataFrame</code> methods like <a href="../reference/api/pandas.DataFrame.sum.html#pandas.DataFrame.sum" title="pandas.DataFrame.sum"><code>DataFrame.sum()</code></a> also show “unused” categories when <code>observed=False</code>.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell40"><span><span class="gp">In [132]: <span class="n">columns <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">(
<span class="gp">    <span class="p">[<span class="s2">"One"<span class="p">, <span class="s2">"One"<span class="p">, <span class="s2">"Two"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"One"<span class="p">, <span class="s2">"Two"<span class="p">, <span class="s2">"Three"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True
<span class="gp"><span class="p">)
<span class="gp">

<span class="gp">In [133]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">(
<span class="gp">    <span class="n">data<span class="o">=<span class="p">[[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">], <span class="p">[<span class="mi">4<span class="p">, <span class="mi">5<span class="p">, <span class="mi">6<span class="p">]],
<span class="gp">    <span class="n">columns<span class="o">=<span class="n">pd<span class="o">.<span class="n">MultiIndex<span class="o">.<span class="n">from_arrays<span class="p">([[<span class="s2">"A"<span class="p">, <span class="s2">"B"<span class="p">, <span class="s2">"B"<span class="p">], <span class="n">columns<span class="p">]),
<span class="gp"><span class="p">)<span class="o">.<span class="n">T
<span class="gp">

<span class="gp">In [134]: <span class="n">df<span class="o">.<span class="n">groupby<span class="p">(<span class="n">level<span class="o">=<span class="mi">1<span class="p">, <span class="n">observed<span class="o">=<span class="kc">False<span class="p">)<span class="o">.<span class="n">sum<span class="p">()
<span class="gh">Out[134]:
<span class="go">       0  1
<span class="go">One    3  9
<span class="go">Two    3  6
<span class="go">Three  0  0
</pre>
</div>
</div>

<p>Groupby will also show “unused” categories when <code>observed=False</code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell41"><span><span class="gp">In [135]: <span class="n">cats <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">(
<span class="gp">    <span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"c"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"d"<span class="p">]
<span class="gp"><span class="p">)
<span class="gp">

<span class="gp">In [136]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">({<span class="s2">"cats"<span class="p">: <span class="n">cats<span class="p">, <span class="s2">"values"<span class="p">: <span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">2<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">, <span class="mi">5<span class="p">]})

<span class="gp">In [137]: <span class="n">df<span class="o">.<span class="n">groupby<span class="p">(<span class="s2">"cats"<span class="p">, <span class="n">observed<span class="o">=<span class="kc">False<span class="p">)<span class="o">.<span class="n">mean<span class="p">()
<span class="gh">Out[137]:
<span class="go">      values
<span class="go">cats        
<span class="go">a        1.0
<span class="go">b        2.0
<span class="go">c        4.0
<span class="go">d        NaN

<span class="gp">In [138]: <span class="n">cats2 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">])

<span class="gp">In [139]: <span class="n">df2 <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">(
<span class="gp">    <span class="p">{
<span class="gp">        <span class="s2">"cats"<span class="p">: <span class="n">cats2<span class="p">,
<span class="gp">        <span class="s2">"B"<span class="p">: <span class="p">[<span class="s2">"c"<span class="p">, <span class="s2">"d"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"d"<span class="p">],
<span class="gp">        <span class="s2">"values"<span class="p">: <span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">],
<span class="gp">    <span class="p">}
<span class="gp"><span class="p">)
<span class="gp">

<span class="gp">In [140]: <span class="n">df2<span class="o">.<span class="n">groupby<span class="p">([<span class="s2">"cats"<span class="p">, <span class="s2">"B"<span class="p">], <span class="n">observed<span class="o">=<span class="kc">False<span class="p">)<span class="o">.<span class="n">mean<span class="p">()
<span class="gh">Out[140]:
<span class="go">        values
<span class="go">cats B        
<span class="go">a    c     1.0
<span class="go">     d     2.0
<span class="go">b    c     3.0
<span class="go">     d     4.0
<span class="go">c    c     NaN
<span class="go">     d     NaN
</pre>
</div>
</div>

<p>Pivot tables:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell42"><span><span class="gp">In [141]: <span class="n">raw_cat <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">])

<span class="gp">In [142]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">({<span class="s2">"A"<span class="p">: <span class="n">raw_cat<span class="p">, <span class="s2">"B"<span class="p">: <span class="p">[<span class="s2">"c"<span class="p">, <span class="s2">"d"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"d"<span class="p">], <span class="s2">"values"<span class="p">: <span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">]})

<span class="gp">In [143]: <span class="n">pd<span class="o">.<span class="n">pivot_table<span class="p">(<span class="n">df<span class="p">, <span class="n">values<span class="o">=<span class="s2">"values"<span class="p">, <span class="n">index<span class="o">=<span class="p">[<span class="s2">"A"<span class="p">, <span class="s2">"B"<span class="p">], <span class="n">observed<span class="o">=<span class="kc">False<span class="p">)
<span class="gh">Out[143]:
<span class="go">     values
<span class="go">A B        
<span class="go">a c     1.0
<span class="go">  d     2.0
<span class="go">b c     3.0
<span class="go">  d     4.0
</pre>
</div>
</div>

# <h2>Data munging</h2>

<p>The optimized pandas data access methods  <code>.loc</code>, <code>.iloc</code>, <code>.at</code>, and <code>.iat</code>, work as normal. The only difference is the return type (for getting) and that only values already in <code>categories</code> can be assigned.</p>

## <h3>Getting</h3>

<p>If the slicing operation returns either a <code>DataFrame</code> or a column of type <code>Series</code>, the <code>category</code> dtype is preserved.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell43"><span><span class="gp">In [144]: <span class="n">idx <span class="o">= <span class="n">pd<span class="o">.<span class="n">Index<span class="p">([<span class="s2">"h"<span class="p">, <span class="s2">"i"<span class="p">, <span class="s2">"j"<span class="p">, <span class="s2">"k"<span class="p">, <span class="s2">"l"<span class="p">, <span class="s2">"m"<span class="p">, <span class="s2">"n"<span class="p">])

<span class="gp">In [145]: <span class="n">cats <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"c"<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">, <span class="n">index<span class="o">=<span class="n">idx<span class="p">)

<span class="gp">In [146]: <span class="n">values <span class="o">= <span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">2<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">, <span class="mi">5<span class="p">]

<span class="gp">In [147]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">({<span class="s2">"cats"<span class="p">: <span class="n">cats<span class="p">, <span class="s2">"values"<span class="p">: <span class="n">values<span class="p">}, <span class="n">index<span class="o">=<span class="n">idx<span class="p">)

<span class="gp">In [148]: <span class="n">df<span class="o">.<span class="n">iloc<span class="p">[<span class="mi">2<span class="p">:<span class="mi">4<span class="p">, <span class="p">:]
<span class="gh">Out[148]:
<span class="go">  cats  values
<span class="go">j    b       2
<span class="go">k    b       2

<span class="gp">In [149]: <span class="n">df<span class="o">.<span class="n">iloc<span class="p">[<span class="mi">2<span class="p">:<span class="mi">4<span class="p">, <span class="p">:]<span class="o">.<span class="n">dtypes
<span class="gh">Out[149]:
<span class="go">cats      category
<span class="go">values       int64
<span class="go">dtype: object

<span class="gp">In [150]: <span class="n">df<span class="o">.<span class="n">loc<span class="p">[<span class="s2">"h"<span class="p">:<span class="s2">"j"<span class="p">, <span class="s2">"cats"<span class="p">]
<span class="gh">Out[150]:
<span class="go">h    a
<span class="go">i    b
<span class="go">j    b
<span class="go">Name: cats, dtype: category
<span class="go">Categories (3, object): ['a', 'b', 'c']

<span class="gp">In [151]: <span class="n">df<span class="p">[<span class="n">df<span class="p">[<span class="s2">"cats"<span class="p">] <span class="o">== <span class="s2">"b"<span class="p">]
<span class="gh">Out[151]:
<span class="go">  cats  values
<span class="go">i    b       2
<span class="go">j    b       2
<span class="go">k    b       2
</pre>
</div>
</div>

<p>An example where the category type is not preserved is if you take one single
row: the resulting <code>Series</code> is of dtype <code>object</code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell44"><span><span class="go"># get the complete "h" row as a Series
<span class="gp">In [152]: <span class="n">df<span class="o">.<span class="n">loc<span class="p">[<span class="s2">"h"<span class="p">, <span class="p">:]
<span class="gh">Out[152]:
<span class="go">cats      a
<span class="go">values    1
<span class="go">Name: h, dtype: object
</pre>
</div>
</div>

<p>Returning a single item from categorical data will also return the value, not a categorical
of length “1”.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell45"><span><span class="gp">In [153]: <span class="n">df<span class="o">.<span class="n">iat<span class="p">[<span class="mi">0<span class="p">, <span class="mi">0<span class="p">]
<span class="gh">Out[153]: <span class="go">'a'

<span class="gp">In [154]: <span class="n">df<span class="p">[<span class="s2">"cats"<span class="p">] <span class="o">= <span class="n">df<span class="p">[<span class="s2">"cats"<span class="p">]<span class="o">.<span class="n">cat<span class="o">.<span class="n">rename_categories<span class="p">([<span class="s2">"x"<span class="p">, <span class="s2">"y"<span class="p">, <span class="s2">"z"<span class="p">])

<span class="gp">In [155]: <span class="n">df<span class="o">.<span class="n">at<span class="p">[<span class="s2">"h"<span class="p">, <span class="s2">"cats"<span class="p">]  <span class="c1"># returns a string
<span class="gh">Out[155]: <span class="go">'x'
</pre>
</div>
</div>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>The is in contrast to R’s <code>factor</code> function, where <code>factor(c(1,2,3))[1]</code>
returns a single value <code>factor</code>.</p>
</div>
<p>To get a single value <code>Series</code> of type <code>category</code>, you pass in a list with
a single value:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell46"><span><span class="gp">In [156]: <span class="n">df<span class="o">.<span class="n">loc<span class="p">[[<span class="s2">"h"<span class="p">], <span class="s2">"cats"<span class="p">]
<span class="gh">Out[156]:
<span class="go">h    x
<span class="go">Name: cats, dtype: category
<span class="go">Categories (3, object): ['x', 'y', 'z']
</pre>
</div>
</div>

## <h3>String and datetime accessors</h3>

<p>The accessors  <code>.dt</code> and <code>.str</code> will work if the <code>s.cat.categories</code> are of
an appropriate type:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell47"><span><span class="gp">In [157]: <span class="n">str_s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">(<span class="nb">list<span class="p">(<span class="s2">"aabb"<span class="p">))

<span class="gp">In [158]: <span class="n">str_cat <span class="o">= <span class="n">str_s<span class="o">.<span class="n">astype<span class="p">(<span class="s2">"category"<span class="p">)

<span class="gp">In [159]: <span class="n">str_cat
<span class="gh">Out[159]:
<span class="go">0    a
<span class="go">1    a
<span class="go">2    b
<span class="go">3    b
<span class="go">dtype: category
<span class="go">Categories (2, object): ['a', 'b']

<span class="gp">In [160]: <span class="n">str_cat<span class="o">.<span class="n">str<span class="o">.<span class="n">contains<span class="p">(<span class="s2">"a"<span class="p">)
<span class="gh">Out[160]:
<span class="go">0     True
<span class="go">1     True
<span class="go">2    False
<span class="go">3    False
<span class="go">dtype: bool

<span class="gp">In [161]: <span class="n">date_s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">(<span class="n">pd<span class="o">.<span class="n">date_range<span class="p">(<span class="s2">"1/1/2015"<span class="p">, <span class="n">periods<span class="o">=<span class="mi">5<span class="p">))

<span class="gp">In [162]: <span class="n">date_cat <span class="o">= <span class="n">date_s<span class="o">.<span class="n">astype<span class="p">(<span class="s2">"category"<span class="p">)

<span class="gp">In [163]: <span class="n">date_cat
<span class="gh">Out[163]:
<span class="go">0   2015-01-01
<span class="go">1   2015-01-02
<span class="go">2   2015-01-03
<span class="go">3   2015-01-04
<span class="go">4   2015-01-05
<span class="go">dtype: category
<span class="go">Categories (5, datetime64[ns]): [2015-01-01, 2015-01-02, 2015-01-03, 2015-01-04, 2015-01-05]

<span class="gp">In [164]: <span class="n">date_cat<span class="o">.<span class="n">dt<span class="o">.<span class="n">day
<span class="gh">Out[164]:
<span class="go">0    1
<span class="go">1    2
<span class="go">2    3
<span class="go">3    4
<span class="go">4    5
<span class="go">dtype: int32
</pre>
</div>
</div>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>The returned <code>Series</code> (or <code>DataFrame</code>) is of the same type as if you used the
<code>.str.&lt;method&gt;</code> / <code>.dt.&lt;method&gt;</code> on a <code>Series</code> of that type (and not of
type <code>category</code>!).</p>
</div>

<p>That means, that the returned values from methods and properties on the accessors of a <code>Series</code> and the returned values from methods and properties on the accessors of this <code>Series</code> transformed to one of type <code>category</code> will be equal:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell48"><span><span class="gp">In [165]: <span class="n">ret_s <span class="o">= <span class="n">str_s<span class="o">.<span class="n">str<span class="o">.<span class="n">contains<span class="p">(<span class="s2">"a"<span class="p">)

<span class="gp">In [166]: <span class="n">ret_cat <span class="o">= <span class="n">str_cat<span class="o">.<span class="n">str<span class="o">.<span class="n">contains<span class="p">(<span class="s2">"a"<span class="p">)

<span class="gp">In [167]: <span class="n">ret_s<span class="o">.<span class="n">dtype <span class="o">== <span class="n">ret_cat<span class="o">.<span class="n">dtype
<span class="gh">Out[167]: <span class="go">True

<span class="gp">In [168]: <span class="n">ret_s <span class="o">== <span class="n">ret_cat
<span class="gh">Out[168]:
<span class="go">0    True
<span class="go">1    True
<span class="go">2    True
<span class="go">3    True
<span class="go">dtype: bool
</pre>
</div>
</div>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>The work is done on the <code>categories</code> and then a new <code>Series</code> is constructed. This has
some performance implication if you have a <code>Series</code> of type string, where lots of elements
are repeated (i.e. the number of unique elements in the <code>Series</code> is a lot smaller than the
length of the <code>Series</code>). In this case it can be faster to convert the original <code>Series</code>
to one of type <code>category</code> and use <code>.str.&lt;method&gt;</code> or <code>.dt.&lt;property&gt;</code> on that.</p>
</div>

## <h3>Setting</h3>

<p>Setting values in a categorical column (or <code>Series</code>) works as long as the value is included in the <code>categories</code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell49"><span><span class="gp">In [169]: <span class="n">idx <span class="o">= <span class="n">pd<span class="o">.<span class="n">Index<span class="p">([<span class="s2">"h"<span class="p">, <span class="s2">"i"<span class="p">, <span class="s2">"j"<span class="p">, <span class="s2">"k"<span class="p">, <span class="s2">"l"<span class="p">, <span class="s2">"m"<span class="p">, <span class="s2">"n"<span class="p">])

<span class="gp">In [170]: <span class="n">cats <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">])

<span class="gp">In [171]: <span class="n">values <span class="o">= <span class="p">[<span class="mi">1<span class="p">, <span class="mi">1<span class="p">, <span class="mi">1<span class="p">, <span class="mi">1<span class="p">, <span class="mi">1<span class="p">, <span class="mi">1<span class="p">, <span class="mi">1<span class="p">]

<span class="gp">In [172]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">({<span class="s2">"cats"<span class="p">: <span class="n">cats<span class="p">, <span class="s2">"values"<span class="p">: <span class="n">values<span class="p">}, <span class="n">index<span class="o">=<span class="n">idx<span class="p">)

<span class="gp">In [173]: <span class="n">df<span class="o">.<span class="n">iloc<span class="p">[<span class="mi">2<span class="p">:<span class="mi">4<span class="p">, <span class="p">:] <span class="o">= <span class="p">[[<span class="s2">"b"<span class="p">, <span class="mi">2<span class="p">], <span class="p">[<span class="s2">"b"<span class="p">, <span class="mi">2<span class="p">]]

<span class="gp">In [174]: <span class="n">df
<span class="gh">Out[174]:
<span class="go">  cats  values
<span class="go">h    a       1
<span class="go">i    a       1
<span class="go">j    b       2
<span class="go">k    b       2
<span class="go">l    a       1
<span class="go">m    a       1
<span class="go">n    a       1

<span class="gp">In [175]: <span class="k">try<span class="p">:
<span class="gp">    <span class="n">df<span class="o">.<span class="n">iloc<span class="p">[<span class="mi">2<span class="p">:<span class="mi">4<span class="p">, <span class="p">:] <span class="o">= <span class="p">[[<span class="s2">"c"<span class="p">, <span class="mi">3<span class="p">], <span class="p">[<span class="s2">"c"<span class="p">, <span class="mi">3<span class="p">]]
<span class="gp"><span class="k">except <span class="ne">TypeError <span class="k">as <span class="n">e<span class="p">:
<span class="gp">    <span class="nb">print<span class="p">(<span class="s2">"TypeError:"<span class="p">, <span class="nb">str<span class="p">(<span class="n">e<span class="p">))
<span class="gp">
<span class="go">TypeError: Cannot setitem on a Categorical with a new category, set the categories first
</pre>
</div>
</div>

<p>Setting values by assigning categorical data will also check that the <code>categories</code> match:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell50"><span><span class="gp">In [176]: <span class="n">df<span class="o">.<span class="n">loc<span class="p">[<span class="s2">"j"<span class="p">:<span class="s2">"k"<span class="p">, <span class="s2">"cats"<span class="p">] <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">])

<span class="gp">In [177]: <span class="n">df
<span class="gh">Out[177]:
<span class="go">  cats  values
<span class="go">h    a       1
<span class="go">i    a       1
<span class="go">j    a       2
<span class="go">k    a       2
<span class="go">l    a       1
<span class="go">m    a       1
<span class="go">n    a       1

<span class="gp">In [178]: <span class="k">try<span class="p">:
<span class="gp">    <span class="n">df<span class="o">.<span class="n">loc<span class="p">[<span class="s2">"j"<span class="p">:<span class="s2">"k"<span class="p">, <span class="s2">"cats"<span class="p">] <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">])
<span class="gp"><span class="k">except <span class="ne">TypeError <span class="k">as <span class="n">e<span class="p">:
<span class="gp">    <span class="nb">print<span class="p">(<span class="s2">"TypeError:"<span class="p">, <span class="nb">str<span class="p">(<span class="n">e<span class="p">))
<span class="gp">
<span class="go">TypeError: Cannot set a Categorical with another, without identical categories
</pre>
</div>
</div>
<p>Assigning a <code>Categorical</code> to parts of a column of other types will use the values:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell51"><span><span class="gp">In [179]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">({<span class="s2">"a"<span class="p">: <span class="p">[<span class="mi">1<span class="p">, <span class="mi">1<span class="p">, <span class="mi">1<span class="p">, <span class="mi">1<span class="p">, <span class="mi">1<span class="p">], <span class="s2">"b"<span class="p">: <span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">]})

<span class="gp">In [180]: <span class="n">df<span class="o">.<span class="n">loc<span class="p">[<span class="mi">1<span class="p">:<span class="mi">2<span class="p">, <span class="s2">"a"<span class="p">] <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">])

<span class="gp">In [181]: <span class="n">df<span class="o">.<span class="n">loc<span class="p">[<span class="mi">2<span class="p">:<span class="mi">3<span class="p">, <span class="s2">"b"<span class="p">] <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">])

<span class="gp">In [182]: <span class="n">df
<span class="gh">Out[182]:
<span class="go">   a  b
<span class="go">0  1  a
<span class="go">1  b  a
<span class="go">2  b  b
<span class="go">3  1  b
<span class="go">4  1  a

<span class="gp">In [183]: <span class="n">df<span class="o">.<span class="n">dtypes
<span class="gh">Out[183]:
<span class="go">a    object
<span class="go">b    object
<span class="go">dtype: object
</pre>
</div>
</div>

## <h3>Merging / concatenation</h3>

<p>By default, combining <code>Series</code> or <code>DataFrames</code> which contain the same
categories results in <code>category</code> dtype, otherwise results will depend on the
dtype of the underlying categories. Merges that result in non-categorical
dtypes will likely have higher memory usage. Use <code>.astype</code> or
<code>union_categoricals</code> to ensure <code>category</code> results.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell52"><span><span class="gp">In [184]: <span class="kn">from<span class="w"> <span class="nn">pandas.api.types<span class="w"> <span class="kn">import <span class="n">union_categoricals

<span class="go"># same categories
<span class="gp">In [185]: <span class="n">s1 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="gp">In [186]: <span class="n">s2 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"a"<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="gp">In [187]: <span class="n">pd<span class="o">.<span class="n">concat<span class="p">([<span class="n">s1<span class="p">, <span class="n">s2<span class="p">])
<span class="gh">Out[187]:
<span class="go">0    a
<span class="go">1    b
<span class="go">0    a
<span class="go">1    b
<span class="go">2    a
<span class="go">dtype: category
<span class="go">Categories (2, object): ['a', 'b']

<span class="go"># different categories
<span class="gp">In [188]: <span class="n">s3 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="gp">In [189]: <span class="n">pd<span class="o">.<span class="n">concat<span class="p">([<span class="n">s1<span class="p">, <span class="n">s3<span class="p">])
<span class="gh">Out[189]:
<span class="go">0    a
<span class="go">1    b
<span class="go">0    b
<span class="go">1    c
<span class="go">dtype: object

<span class="go"># Output dtype is inferred based on categories values
<span class="gp">In [190]: <span class="n">int_cats <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="gp">In [191]: <span class="n">float_cats <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="mf">3.0<span class="p">, <span class="mf">4.0<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="gp">In [192]: <span class="n">pd<span class="o">.<span class="n">concat<span class="p">([<span class="n">int_cats<span class="p">, <span class="n">float_cats<span class="p">])
<span class="gh">Out[192]:
<span class="go">0    1.0
<span class="go">1    2.0
<span class="go">0    3.0
<span class="go">1    4.0
<span class="go">dtype: float64

<span class="gp">In [193]: <span class="n">pd<span class="o">.<span class="n">concat<span class="p">([<span class="n">s1<span class="p">, <span class="n">s3<span class="p">])<span class="o">.<span class="n">astype<span class="p">(<span class="s2">"category"<span class="p">)
<span class="gh">Out[193]:
<span class="go">0    a
<span class="go">1    b
<span class="go">0    b
<span class="go">1    c
<span class="go">dtype: category
<span class="go">Categories (3, object): ['a', 'b', 'c']

<span class="gp">In [194]: <span class="n">union_categoricals<span class="p">([<span class="n">s1<span class="o">.<span class="n">array<span class="p">, <span class="n">s3<span class="o">.<span class="n">array<span class="p">])
<span class="gh">Out[194]:
<span class="go">['a', 'b', 'b', 'c']
<span class="go">Categories (3, object): ['a', 'b', 'c']
</pre>
</div>
</div>

<p>The following table summarizes the results of merging <code>Categoricals</code>:</p>
<table class="table">
<thead>
<tr class="row-odd"><th class="head"><p>arg1</p></th>
<th class="head"><p>arg2</p></th>
<th class="head"><p>identical</p></th>
<th class="head"><p>result</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p>category</p></td>
<td><p>category</p></td>
<td><p>True</p></td>
<td><p>category</p></td>
</tr>
<tr class="row-odd"><td><p>category (object)</p></td>
<td><p>category (object)</p></td>
<td><p>False</p></td>
<td><p>object (dtype is inferred)</p></td>
</tr>
<tr class="row-even"><td><p>category (int)</p></td>
<td><p>category (float)</p></td>
<td><p>False</p></td>
<td><p>float (dtype is inferred)</p></td>
</tr>
</tbody>
</table>

## <h3>Unioning</h3>

<p>If you want to combine categoricals that do not necessarily have the same categories, the <a href="../reference/api/pandas.api.types.union_categoricals.html#pandas.api.types.union_categoricals" title="pandas.api.types.union_categoricals"><code>union_categoricals()</code></a> function will combine a list-like of categoricals.The new categories will be the union of the categories being combined.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell53"><span><span class="gp">In [195]: <span class="kn">from<span class="w"> <span class="nn">pandas.api.types<span class="w"> <span class="kn">import <span class="n">union_categoricals

<span class="gp">In [196]: <span class="n">a <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">])

<span class="gp">In [197]: <span class="n">b <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">])

<span class="gp">In [198]: <span class="n">union_categoricals<span class="p">([<span class="n">a<span class="p">, <span class="n">b<span class="p">])
<span class="gh">Out[198]:
<span class="go">['b', 'c', 'a', 'b']
<span class="go">Categories (3, object): ['b', 'c', 'a']
</pre>
</div>
</div>

<p>By default, the resulting categories will be ordered as they appear in the data. If you want the categories to be lexsorted, use <code>sort_categories=True</code> argument.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell54"><span><span class="gp">In [199]: <span class="n">union_categoricals<span class="p">([<span class="n">a<span class="p">, <span class="n">b<span class="p">], <span class="n">sort_categories<span class="o">=<span class="kc">True<span class="p">)
<span class="gh">Out[199]:
<span class="go">['b', 'c', 'a', 'b']
<span class="go">Categories (3, object): ['a', 'b', 'c']
</pre>
</div>
</div>

<p><code>union_categoricals</code> also works with the “easy” case of combining two categoricals of the same categories and order information (e.g. what you could also <code>append</code> for).</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell55"><span><span class="gp">In [200]: <span class="n">a <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">)

<span class="gp">In [201]: <span class="n">b <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"a"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">)

<span class="gp">In [202]: <span class="n">union_categoricals<span class="p">([<span class="n">a<span class="p">, <span class="n">b<span class="p">])
<span class="gh">Out[202]:
<span class="go">['a', 'b', 'a', 'b', 'a']
<span class="go">Categories (2, object): ['a' &lt; 'b']
</pre>
</div>
</div>

<p>The below raises <code>TypeError</code> because the categories are ordered and not identical.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell56"><span><span class="gp">In [203]: <span class="n">a <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">)

<span class="gp">In [204]: <span class="n">b <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">)

<span class="gp">In [205]: <span class="n">union_categoricals<span class="p">([<span class="n">a<span class="p">, <span class="n">b<span class="p">])
<span class="gt">---------------------------------------------------------------------------
<span class="ne">TypeError<span class="g g-Whitespace">                                 Traceback (most recent call last)
<span class="n">Cell <span class="n">In<span class="p">[<span class="mi">205<span class="p">], <span class="n">line <span class="mi">1
<span class="ne">----&gt; <span class="mi">1 <span class="n">union_categoricals<span class="p">([<span class="n">a<span class="p">, <span class="n">b<span class="p">])

<span class="nn">File ~/work/pandas/pandas/pandas/core/dtypes/concat.py:341, in <span class="ni">union_categoricals<span class="nt">(to_union, sort_categories, ignore_order)
<span class="g g-Whitespace">    <span class="mi">339     <span class="k">if <span class="nb">all<span class="p">(<span class="n">c<span class="o">.<span class="n">ordered <span class="k">for <span class="n">c <span class="ow">in <span class="n">to_union<span class="p">):
<span class="g g-Whitespace">    <span class="mi">340         <span class="n">msg <span class="o">= <span class="s2">"to union ordered Categoricals, all categories must be the same"
<span class="ne">--&gt; <span class="mi">341         <span class="k">raise <span class="ne">TypeError<span class="p">(<span class="n">msg<span class="p">)
<span class="g g-Whitespace">    <span class="mi">342     <span class="k">raise <span class="ne">TypeError<span class="p">(<span class="s2">"Categorical.ordered must be the same"<span class="p">)
<span class="g g-Whitespace">    <span class="mi">344 <span class="k">if <span class="n">ignore_order<span class="p">:

<span class="ne">TypeError: to union ordered Categoricals, all categories must be the same
</pre>
</div>
</div>

<p>Ordered categoricals with different categories or orderings can be combined by using the <code>ignore_ordered=True</code> argument.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell57"><span><span class="gp">In [206]: <span class="n">a <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">)

<span class="gp">In [207]: <span class="n">b <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"c"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"a"<span class="p">], <span class="n">ordered<span class="o">=<span class="kc">True<span class="p">)

<span class="gp">In [208]: <span class="n">union_categoricals<span class="p">([<span class="n">a<span class="p">, <span class="n">b<span class="p">], <span class="n">ignore_order<span class="o">=<span class="kc">True<span class="p">)
<span class="gh">Out[208]:
<span class="go">['a', 'b', 'c', 'c', 'b', 'a']
<span class="go">Categories (3, object): ['a', 'b', 'c']
</pre>
</div>
</div>
<p><a href="../reference/api/pandas.api.types.union_categoricals.html#pandas.api.types.union_categoricals" title="pandas.api.types.union_categoricals"><code>union_categoricals()</code></a> also works with a
<code>CategoricalIndex</code>, or <code>Series</code> containing categorical data, but note that
the resulting array will always be a plain <code>Categorical</code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell58"><span><span class="gp">In [209]: <span class="n">a <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="gp">In [210]: <span class="n">b <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="gp">In [211]: <span class="n">union_categoricals<span class="p">([<span class="n">a<span class="p">, <span class="n">b<span class="p">])
<span class="gh">Out[211]:
<span class="go">['b', 'c', 'a', 'b']
<span class="go">Categories (3, object): ['b', 'c', 'a']
</pre>
</div>
</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p><code>union_categoricals</code> may recode the integer codes for categories
when combining categoricals.  This is likely what you want,
but if you are relying on the exact numbering of the categories, be
aware.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell59"><span><span class="gp">In [212]: <span class="n">c1 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">])

<span class="gp">In [213]: <span class="n">c2 <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">])

<span class="gp">In [214]: <span class="n">c1
<span class="gh">Out[214]:
<span class="go">['b', 'c']
<span class="go">Categories (2, object): ['b', 'c']

<span class="go"># "b" is coded to 0
<span class="gp">In [215]: <span class="n">c1<span class="o">.<span class="n">codes
<span class="gh">Out[215]: <span class="go">array([0, 1], dtype=int8)

<span class="gp">In [216]: <span class="n">c2
<span class="gh">Out[216]:
<span class="go">['a', 'b']
<span class="go">Categories (2, object): ['a', 'b']

<span class="go"># "b" is coded to 1
<span class="gp">In [217]: <span class="n">c2<span class="o">.<span class="n">codes
<span class="gh">Out[217]: <span class="go">array([0, 1], dtype=int8)

<span class="gp">In [218]: <span class="n">c <span class="o">= <span class="n">union_categoricals<span class="p">([<span class="n">c1<span class="p">, <span class="n">c2<span class="p">])

<span class="gp">In [219]: <span class="n">c
<span class="gh">Out[219]:
<span class="go">['b', 'c', 'a', 'b']
<span class="go">Categories (3, object): ['b', 'c', 'a']

<span class="go"># "b" is coded to 0 throughout, same as c1, different from c2
<span class="gp">In [220]: <span class="n">c<span class="o">.<span class="n">codes
<span class="gh">Out[220]: <span class="go">array([0, 1, 2, 0], dtype=int8)
</pre>
</div>
</div>
</div>

# <h2>Getting data in/out</h2>

<p>You can write data that contains <code>category</code> dtypes to a <code>HDFStore</code>.
See <a href="io.html#io-hdf5-categorical">here</a> for an example and caveats.</p>

<p>It is also possible to write data to and reading data from <em>Stata</em> format files.
See <a href="io.html#io-stata-categorical">here</a> for an example and caveats.</p>

<p>Writing to a CSV file will convert the data, effectively removing any information about the categorical (categories and ordering). So if you read back the CSV file you have to convert the relevant columns back to <code>category</code> and assign the right categories and categories ordering.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell60"><span><span class="gp">In [221]: <span class="kn">import<span class="w"> <span class="nn">io

<span class="gp">In [222]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">(<span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"a"<span class="p">, <span class="s2">"d"<span class="p">]))

<span class="go"># rename the categories
<span class="gp">In [223]: <span class="n">s <span class="o">= <span class="n">s<span class="o">.<span class="n">cat<span class="o">.<span class="n">rename_categories<span class="p">([<span class="s2">"very good"<span class="p">, <span class="s2">"good"<span class="p">, <span class="s2">"bad"<span class="p">])

<span class="go"># reorder the categories and add missing categories
<span class="gp">In [224]: <span class="n">s <span class="o">= <span class="n">s<span class="o">.<span class="n">cat<span class="o">.<span class="n">set_categories<span class="p">([<span class="s2">"very bad"<span class="p">, <span class="s2">"bad"<span class="p">, <span class="s2">"medium"<span class="p">, <span class="s2">"good"<span class="p">, <span class="s2">"very good"<span class="p">])

<span class="gp">In [225]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">({<span class="s2">"cats"<span class="p">: <span class="n">s<span class="p">, <span class="s2">"vals"<span class="p">: <span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">, <span class="mi">5<span class="p">, <span class="mi">6<span class="p">]})

<span class="gp">In [226]: <span class="n">csv <span class="o">= <span class="n">io<span class="o">.<span class="n">StringIO<span class="p">()

<span class="gp">In [227]: <span class="n">df<span class="o">.<span class="n">to_csv<span class="p">(<span class="n">csv<span class="p">)

<span class="gp">In [228]: <span class="n">df2 <span class="o">= <span class="n">pd<span class="o">.<span class="n">read_csv<span class="p">(<span class="n">io<span class="o">.<span class="n">StringIO<span class="p">(<span class="n">csv<span class="o">.<span class="n">getvalue<span class="p">()))

<span class="gp">In [229]: <span class="n">df2<span class="o">.<span class="n">dtypes
<span class="gh">Out[229]:
<span class="go">Unnamed: 0     int64
<span class="go">cats          object
<span class="go">vals           int64
<span class="go">dtype: object

<span class="gp">In [230]: <span class="n">df2<span class="p">[<span class="s2">"cats"<span class="p">]
<span class="gh">Out[230]:
<span class="go">0    very good
<span class="go">1         good
<span class="go">2         good
<span class="go">3    very good
<span class="go">4    very good
<span class="go">5          bad
<span class="go">Name: cats, dtype: object

<span class="go"># Redo the category
<span class="gp">In [231]: <span class="n">df2<span class="p">[<span class="s2">"cats"<span class="p">] <span class="o">= <span class="n">df2<span class="p">[<span class="s2">"cats"<span class="p">]<span class="o">.<span class="n">astype<span class="p">(<span class="s2">"category"<span class="p">)

<span class="gp">In [232]: <span class="n">df2<span class="p">[<span class="s2">"cats"<span class="p">] <span class="o">= <span class="n">df2<span class="p">[<span class="s2">"cats"<span class="p">]<span class="o">.<span class="n">cat<span class="o">.<span class="n">set_categories<span class="p">(
<span class="gp">    <span class="p">[<span class="s2">"very bad"<span class="p">, <span class="s2">"bad"<span class="p">, <span class="s2">"medium"<span class="p">, <span class="s2">"good"<span class="p">, <span class="s2">"very good"<span class="p">]
<span class="gp"><span class="p">)
<span class="gp">

<span class="gp">In [233]: <span class="n">df2<span class="o">.<span class="n">dtypes
<span class="gh">Out[233]:
<span class="go">Unnamed: 0       int64
<span class="go">cats          category
<span class="go">vals             int64
<span class="go">dtype: object

<span class="gp">In [234]: <span class="n">df2<span class="p">[<span class="s2">"cats"<span class="p">]
<span class="gh">Out[234]:
<span class="go">0    very good
<span class="go">1         good
<span class="go">2         good
<span class="go">3    very good
<span class="go">4    very good
<span class="go">5          bad
<span class="go">Name: cats, dtype: category
<span class="go">Categories (5, object): ['very bad', 'bad', 'medium', 'good', 'very good']
</pre>
</div>
</div>

<p>The same holds for writing to a SQL database with <code>to_sql</code>.</p>

# <h2>Missing data</h2>

<p>pandas primarily uses the value <code>np.nan</code> to represent missing data. It is by default not included in computations. See the <a href="missing_data.html#missing-data">Missing Data section</a>.</p>
<p>Missing values should <strong>not</strong> be included in the Categorical’s <code>categories</code>, only in the <code>values</code>.
Instead, it is understood that NaN is different, and is always a possibility.
When working with the Categorical’s <code>codes</code>, missing values will always have a code of <code>-1</code>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell61"><span><span class="gp">In [235]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="n">np<span class="o">.<span class="n">nan<span class="p">, <span class="s2">"a"<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="go"># only two categories
<span class="gp">In [236]: <span class="n">s
<span class="gh">Out[236]:
<span class="go">0      a
<span class="go">1      b
<span class="go">2    NaN
<span class="go">3      a
<span class="go">dtype: category
<span class="go">Categories (2, object): ['a', 'b']

<span class="gp">In [237]: <span class="n">s<span class="o">.<span class="n">cat<span class="o">.<span class="n">codes
<span class="gh">Out[237]:
<span class="go">0    0
<span class="go">1    1
<span class="go">2   -1
<span class="go">3    0
<span class="go">dtype: int8
</pre>
</div>
</div>

<p>Methods for working with missing data, e.g. <a href="../reference/api/pandas.Series.isna.html#pandas.Series.isna" title="pandas.Series.isna"><code>isna()</a>, <a href="../reference/api/pandas.Series.fillna.html#pandas.Series.fillna" title="pandas.Series.fillna"><code>fillna()</code></a>, <a href="../reference/api/pandas.Series.dropna.html#pandas.Series.dropna" title="pandas.Series.dropna"><code>dropna()</code></a>, all work normally:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell62"><span>lass="gp">In [238]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="n">np<span class="o">.<span class="n">nan<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">)

<span class="gp">In [239]: <span class="n">s
<span class="gh">Out[239]:
<span class="go">0      a
<span class="go">1      b
<span class="go">2    NaN
<span class="go">dtype: category
<span class="go">Categories (2, object): ['a', 'b']

<span class="gp">In [240]: <span class="n">pd<span class="o">.<span class="n">isna<span class="p">(<span class="n">s<span class="p">)
<span class="gh">Out[240]:
<span class="go">0    False
<span class="go">1    False
<span class="go">2     True
<span class="go">dtype: bool

<span class="gp">In [241]: <span class="n">s<span class="o">.<span class="n">fillna<span class="p">(<span class="s2">"a"<span class="p">)
<span class="gh">Out[241]:
<span class="go">0    a
<span class="go">1    b
<span class="go">2    a
<span class="go">dtype: category
<span class="go">Categories (2, object): ['a', 'b']
</pre>
</div>
</div>

# <h2>Differences to R’s <code>factor</h2>

<p>The following differences to R’s factor functions can be observed:</p>

<ul>

<li><p>R’s <code>levels</code> are named <code>categories</code>.</p></li>

<li><p>R’s <code>levels</code> are always of type string, while <code>categories</code> in pandas can be of any dtype.</p></li>

<li><p>It’s not possible to specify labels at creation time. Use <code>s.cat.rename_categories(new_labels)</code>
afterwards.</p></li>

<li><p>In contrast to R’s <code>factor</code> function, using categorical data as the sole input to create a new categorical series will <em>not</em> remove unused categories but create a new categorical series which is equal to the passed in one!</p></li>

<li><p>R allows for missing values to be included in its <code>levels</code> (pandas’ <code>categories</code>). Pandas
does not allow <code>NaN</code> categories, but missing values can still be in the <code>values</code>.</p></li>

</ul>

# <h2>Gotchas</h2>

## <h3>Memory usage</h3>

<p>The memory usage of a <code>Categorical is proportional to the number of categories plus the length of the data. In contrast, an <code>object</code> dtype is a constant times the length of the data.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell63"><span><span class="gp">In [242]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"foo"<span class="p">, <span class="s2">"bar"<span class="p">] <span class="o">* <span class="mi">1000<span class="p">)

<span class="go"># object dtype
<span class="gp">In [243]: <span class="n">s<span class="o">.<span class="n">nbytes
<span class="gh">Out[243]: <span class="go">16000

<span class="go"># category dtype
<span class="gp">In [244]: <span class="n">s<span class="o">.<span class="n">astype<span class="p">(<span class="s2">"category"<span class="p">)<span class="o">.<span class="n">nbytes
<span class="gh">Out[244]: <span class="go">2016
</pre>
</div>
</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>If the number of categories approaches the length of the data, the <code>Categorical</code> will use nearly the same or
more memory than an equivalent <code>object</code> dtype representation.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell64"><span><span class="gp">In [245]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"foo<span class="si">%04d<span class="s2">" <span class="o">% <span class="n">i <span class="k">for <span class="n">i <span class="ow">in <span class="nb">range<span class="p">(<span class="mi">2000<span class="p">)])

<span class="go"># object dtype
<span class="gp">In [246]: <span class="n">s<span class="o">.<span class="n">nbytes
<span class="gh">Out[246]: <span class="go">16000

<span class="go"># category dtype
<span class="gp">In [247]: <span class="n">s<span class="o">.<span class="n">astype<span class="p">(<span class="s2">"category"<span class="p">)<span class="o">.<span class="n">nbytes
<span class="gh">Out[247]: <span class="go">20000
</pre>
</div>
</div>
</div>
</section>
<section id="categorical-is-not-a-numpy-array">
<h3><code>Categorical</code> is not a <code>numpy</code> array<a class="headerlink" href="#categorical-is-not-a-numpy-array" title="Link to this heading">#</a></h3>
<p>Currently, categorical data and the underlying <code>Categorical</code> is implemented as a Python
object and not as a low-level NumPy array dtype. This leads to some problems.</p>
<p>NumPy itself doesn’t know about the new <code>dtype</code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell65"><span><span class="gp">In [248]: <span class="k">try<span class="p">:
<span class="gp">    <span class="n">np<span class="o">.<span class="n">dtype<span class="p">(<span class="s2">"category"<span class="p">)
<span class="gp"><span class="k">except <span class="ne">TypeError <span class="k">as <span class="n">e<span class="p">:
<span class="gp">    <span class="nb">print<span class="p">(<span class="s2">"TypeError:"<span class="p">, <span class="nb">str<span class="p">(<span class="n">e<span class="p">))
<span class="gp">
<span class="go">TypeError: data type 'category' not understood

<span class="gp">In [249]: <span class="n">dtype <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="s2">"a"<span class="p">])<span class="o">.<span class="n">dtype

<span class="gp">In [250]: <span class="k">try<span class="p">:
<span class="gp">    <span class="n">np<span class="o">.<span class="n">dtype<span class="p">(<span class="n">dtype<span class="p">)
<span class="gp"><span class="k">except <span class="ne">TypeError <span class="k">as <span class="n">e<span class="p">:
<span class="gp">    <span class="nb">print<span class="p">(<span class="s2">"TypeError:"<span class="p">, <span class="nb">str<span class="p">(<span class="n">e<span class="p">))
<span class="gp">
<span class="go">TypeError: Cannot interpret 'CategoricalDtype(categories=['a'], ordered=False, categories_dtype=object)' as a data type
</pre>
</div>
</div>
<p>Dtype comparisons work:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell66"><span><span class="gp">In [251]: <span class="n">dtype <span class="o">== <span class="n">np<span class="o">.<span class="n">str_
<span class="gh">Out[251]: <span class="go">False

<span class="gp">In [252]: <span class="n">np<span class="o">.<span class="n">str_ <span class="o">== <span class="n">dtype
<span class="gh">Out[252]: <span class="go">False
</pre>
</div>
</div>
<p>To check if a Series contains Categorical data, use <code>hasattr(s, 'cat')</code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell67"><span><span class="gp">In [253]: <span class="nb">hasattr<span class="p">(<span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"a"<span class="p">], <span class="n">dtype<span class="o">=<span class="s2">"category"<span class="p">), <span class="s2">"cat"<span class="p">)
<span class="gh">Out[253]: <span class="go">True

<span class="gp">In [254]: <span class="nb">hasattr<span class="p">(<span class="n">pd<span class="o">.<span class="n">Series<span class="p">([<span class="s2">"a"<span class="p">]), <span class="s2">"cat"<span class="p">)
<span class="gh">Out[254]: <span class="go">False
</pre>
</div>
</div>
<p>Using NumPy functions on a <code>Series</code> of type <code>category</code> should not work as <code>Categoricals</code>
are not numeric data (even in the case that <code>.categories</code> is numeric).</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell68"><span><span class="gp">In [255]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">(<span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">]))

<span class="gp">In [256]: <span class="k">try<span class="p">:
<span class="gp">    <span class="n">np<span class="o">.<span class="n">sum<span class="p">(<span class="n">s<span class="p">)
<span class="gp"><span class="k">except <span class="ne">TypeError <span class="k">as <span class="n">e<span class="p">:
<span class="gp">    <span class="nb">print<span class="p">(<span class="s2">"TypeError:"<span class="p">, <span class="nb">str<span class="p">(<span class="n">e<span class="p">))
<span class="gp">
<span class="go">TypeError: 'Categorical' with dtype category does not support reduction 'sum'
</pre>
</div>
</div>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>If such a function works, please file a bug at <a class="github reference external" href="https://github.com/pandas-dev/pandas">pandas-dev/pandas</a>!</p>
</div>

## <h3>dtype in apply</h3>

<p>pandas currently does not preserve the dtype in apply functions: If you apply along rows you get a <code>Series</code> of <code>object</code> <code>dtype</code> (same as getting a row -&gt; getting one element will return a basic type) and applying along columns will also convert to object. <code>NaN</code> values are unaffected.
You can use <code>fillna</code> to handle missing values before applying a function.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell69"><span><span class="gp">In [257]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">(
<span class="gp">    <span class="p">{
<span class="gp">        <span class="s2">"a"<span class="p">: <span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">],
<span class="gp">        <span class="s2">"b"<span class="p">: <span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"d"<span class="p">],
<span class="gp">        <span class="s2">"cats"<span class="p">: <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">2<span class="p">]),
<span class="gp">    <span class="p">}
<span class="gp"><span class="p">)
<span class="gp">

<span class="gp">In [258]: <span class="n">df<span class="o">.<span class="n">apply<span class="p">(<span class="k">lambda <span class="n">row<span class="p">: <span class="nb">type<span class="p">(<span class="n">row<span class="p">[<span class="s2">"cats"<span class="p">]), <span class="n">axis<span class="o">=<span class="mi">1<span class="p">)
<span class="gh">Out[258]:
<span class="go">0    &lt;class 'int'&gt;
<span class="go">1    &lt;class 'int'&gt;
<span class="go">2    &lt;class 'int'&gt;
<span class="go">3    &lt;class 'int'&gt;
<span class="go">dtype: object

<span class="gp">In [259]: <span class="n">df<span class="o">.<span class="n">apply<span class="p">(<span class="k">lambda <span class="n">col<span class="p">: <span class="n">col<span class="o">.<span class="n">dtype<span class="p">, <span class="n">axis<span class="o">=<span class="mi">0<span class="p">)
<span class="gh">Out[259]:
<span class="go">a          int64
<span class="go">b         object
<span class="go">cats    category
<span class="go">dtype: object
</pre>
</div>
</div>

## <h3>Categorical index</h3>

<p><code>CategoricalIndex</code> is a type of index that is useful for supporting indexing with duplicates. This is a container around a <code>Categorical</code> and allows efficient indexing and storage of an index with a large number of duplicated elements.
See the <a href="advanced.html#advanced-categoricalindex">advanced indexing docs</a> for a more detailed
explanation.</p>

<p>Setting the index will create a <code>CategoricalIndex</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell70"><span><span class="gp">In [260]: <span class="n">cats <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="mi">4<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">1<span class="p">])

<span class="gp">In [261]: <span class="n">strings <span class="o">= <span class="p">[<span class="s2">"a"<span class="p">, <span class="s2">"b"<span class="p">, <span class="s2">"c"<span class="p">, <span class="s2">"d"<span class="p">]

<span class="gp">In [262]: <span class="n">values <span class="o">= <span class="p">[<span class="mi">4<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">1<span class="p">]

<span class="gp">In [263]: <span class="n">df <span class="o">= <span class="n">pd<span class="o">.<span class="n">DataFrame<span class="p">({<span class="s2">"strings"<span class="p">: <span class="n">strings<span class="p">, <span class="s2">"values"<span class="p">: <span class="n">values<span class="p">}, <span class="n">index<span class="o">=<span class="n">cats<span class="p">)

<span class="gp">In [264]: <span class="n">df<span class="o">.<span class="n">index
<span class="gh">Out[264]: <span class="go">CategoricalIndex([1, 2, 3, 4], categories=[4, 2, 3, 1], ordered=False, dtype='category')

<span class="go"># This now sorts by the categories order
<span class="gp">In [265]: <span class="n">df<span class="o">.<span class="n">sort_index<span class="p">()
<span class="gh">Out[265]:
<span class="go">  strings  values
<span class="go">4       d       1
<span class="go">2       b       2
<span class="go">3       c       3
<span class="go">1       a       4
</pre>
</div>
</div>

## <h3>Side effects</h3>

<p>Constructing a <code>Series</code> from a <code>Categorical</code> will not copy the input <code>Categorical</code>. This means that changes to the <code>Series</code> will in most cases change the original <code>Categorical</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell71"><span><span class="gp">In [266]: <span class="n">cat <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">10<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">, <span class="mi">10<span class="p">])

<span class="gp">In [267]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">(<span class="n">cat<span class="p">, <span class="n">name<span class="o">=<span class="s2">"cat"<span class="p">)

<span class="gp">In [268]: <span class="n">cat
<span class="gh">Out[268]:
<span class="go">[1, 2, 3, 10]
<span class="go">Categories (5, int64): [1, 2, 3, 4, 10]

<span class="gp">In [269]: <span class="n">s<span class="o">.<span class="n">iloc<span class="p">[<span class="mi">0<span class="p">:<span class="mi">2<span class="p">] <span class="o">= <span class="mi">10

<span class="gp">In [270]: <span class="n">cat
<span class="gh">Out[270]:
<span class="go">[10, 10, 3, 10]
<span class="go">Categories (5, int64): [1, 2, 3, 4, 10]
</pre>
</div>
</div>

<p>Use <code>copy=True</code> to prevent such a behaviour or simply don’t reuse <code>Categoricals</code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell72"><span><span class="gp">In [271]: <span class="n">cat <span class="o">= <span class="n">pd<span class="o">.<span class="n">Categorical<span class="p">([<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">10<span class="p">], <span class="n">categories<span class="o">=<span class="p">[<span class="mi">1<span class="p">, <span class="mi">2<span class="p">, <span class="mi">3<span class="p">, <span class="mi">4<span class="p">, <span class="mi">10<span class="p">])

<span class="gp">In [272]: <span class="n">s <span class="o">= <span class="n">pd<span class="o">.<span class="n">Series<span class="p">(<span class="n">cat<span class="p">, <span class="n">name<span class="o">=<span class="s2">"cat"<span class="p">, <span class="n">copy<span class="o">=<span class="kc">True<span class="p">)

<span class="gp">In [273]: <span class="n">cat
<span class="gh">Out[273]:
<span class="go">[1, 2, 3, 10]
<span class="go">Categories (5, int64): [1, 2, 3, 4, 10]

<span class="gp">In [274]: <span class="n">s<span class="o">.<span class="n">iloc<span class="p">[<span class="mi">0<span class="p">:<span class="mi">2<span class="p">] <span class="o">= <span class="mi">10

<span class="gp">In [275]: <span class="n">cat
<span class="gh">Out[275]:
<span class="go">[1, 2, 3, 10]
<span class="go">Categories (5, int64): [1, 2, 3, 4, 10]
</pre>
</div>
</div>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>This also happens in some cases when you supply a NumPy array instead of a <code>Categorical</code>:
using an int array (e.g. <code>np.array([1,2,3,4])</code>) will exhibit the same behavior, while using
a string array (e.g. <code>np.array(["a","b","c","a"])</code>) will not.</p>
</div>